In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()
# TotalCharges is an (int) but in the data (object).


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df["TotalCharges"].sample(n=30)
# This column contains values ​​written as a space, therefore it is considered an object ---> exp ( row numper : 1340 ).


357      6823.4
409        19.6
5263    2479.05
2001     117.95
6365     700.85
1554      518.3
4738      541.9
3043      78.65
6508     4186.3
723       141.5
2434      60.65
3873       1782
120     5515.45
5362     1715.1
5257    3744.05
2100    6841.45
4120    2034.25
39       1105.4
2037       70.6
2014      750.1
5368      235.5
2843     1070.5
1604    6511.25
4268     3165.6
4814     1993.8
4096    1871.15
3308      387.9
1684    1655.35
63        957.1
804      489.45
Name: TotalCharges, dtype: object

In [5]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [6]:
TotalCharges_numeric = pd.to_numeric(df["TotalCharges"] , errors="coerce")

In [7]:
TotalCharges_numeric.isnull().sum()

np.int64(11)

In [8]:
from sklearn.model_selection import train_test_split

train_set = df.drop(columns=["Churn"])
test_set = df["Churn"]

x_train , x_test ,y_train , y_test = train_test_split(train_set , test_set , test_size=0.2 , stratify= test_set , random_state=42)

In [9]:
explore_data = train_set.copy()
explore_data['Churn'] = y_train

In [10]:
print(explore_data.groupby("Contract")["Churn"].value_counts(normalize = True))

Contract        Churn
Month-to-month  No       0.572534
                Yes      0.427466
One year        No       0.889173
                Yes      0.110827
Two year        No       0.971302
                Yes      0.028698
Name: proportion, dtype: float64


In [11]:
print(explore_data[explore_data["TotalCharges"].str.strip() == '' ][['tenure', 'MonthlyCharges', 'TotalCharges']])

      tenure  MonthlyCharges TotalCharges
488        0           52.55             
753        0           20.25             
936        0           80.85             
1082       0           25.75             
1340       0           56.05             
3331       0           19.85             
3826       0           25.35             
4380       0           20.00             
5218       0           19.70             
6670       0           73.35             
6754       0           61.90             


In [12]:
total_serveces = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
            'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

explore_data["totla_serveces"] = explore_data[total_serveces].apply(lambda row : sum((row != "NO") & (row != "No phone service")) , axis= 1)

In [16]:
from sklearn.base import BaseEstimator , TransformerMixin

class TotalChargesTransformer(BaseEstimator , TransformerMixin):
    def __init__(self):
        pass

    def fit(self , x , y=None):
        return self
    
    def transform(self , x):
        x_copy = x.copy()

        if "TotalCharges" in x_copy.columns:
            x_copy["TotalCharges"] = x_copy["TotalCharges"].replace(r'^\s*$' , np.nan , regex=True)
            x_copy["TotalCharges"] = x_copy["TotalCharges"].astype(float)
            x_copy["TotalCharges"] = x_copy["TotalCharges"].fillna(0.0)

        return x_copy
        

In [17]:
class ServiceAggregator (BaseEstimator , TransformerMixin):
    def __init__(self , services_columns):
        self.services_columns = services_columns

    def fit(self , x , y = None):
        return self
    
    def transform (self , x):
        x_copy = x.copy()


        x_copy["total_serveces"] = x_copy[self.services_columns].apply(

            lambda row : sum((row != "No") & (row != "No phone service" )), axis= 1
        )

        if 'MonthlyCharges' in x_copy.columns :
            x_copy["cost_per_secvice"] = x_copy["MonthlyCharges"] / (x_copy["TotalCharges"] + 0.1)

        return x_copy
        

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import make_column_selector

total_serveces = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
                    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']




num_pipeline = Pipeline(steps=[
                          
                          ("scaler" , StandardScaler())
])


cat_pipeline = Pipeline(steps=[

                          ("encoder" , OneHotEncoder(handle_unknown="ignore" , sparse_output=False))
])                            
                               
preprocessing = ColumnTransformer(transformers=[

                          ("num_path" , num_pipeline , make_column_selector(dtype_include=np.number)),
                          ("cat_path" , cat_pipeline , make_column_selector(dtype_include=["object" , "category"]))

])                               


full_pipeline = Pipeline (steps=[

                           ("cleaner" , TotalChargesTransformer()),
                           ("aggregator" , ServiceAggregator(services_columns=total_serveces)),
                           ("preprocessor" , preprocessing),
                           ("classifier" , RandomForestClassifier(random_state=42))
                           
])

full_pipeline.fit(x_train, y_train)

Pipeline(steps=[('cleaner', TotalChargesTransformer()),
                ('aggregator',
                 ServiceAggregator(services_columns=['PhoneService',
                                                     'MultipleLines',
                                                     'InternetService',
                                                     'OnlineSecurity',
                                                     'OnlineBackup',
                                                     'DeviceProtection',
                                                     'TechSupport',
                                                     'StreamingTV',
                                                     'StreamingMovies'])),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num_path',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x000001F6C0590A90>),
                                                 ('cat_path',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x000001F6C0590730>)])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [21]:
from sklearn.metrics import classification_report

y_pred = full_pipeline.predict(x_test)

y_proba = full_pipeline.predict_proba(x_test)[: , 1]
print("===== Classification report =====")
print(classification_report(y_test , y_pred))

===== Classification report =====
              precision    recall  f1-score   support

          No       0.83      0.91      0.87      1035
         Yes       0.66      0.49      0.56       374

    accuracy                           0.80      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.79      0.80      0.79      1409



In [22]:
from sklearn.model_selection import GridSearchCV
 
param_grid = {
            
            "classifier__n_estimators" : [100 , 200 , 300] , 
            "classifier__max_depth" : [10 , 15 , 20 , None], 
            "classifier__class_weight" : ['balanced' , 'balanced_subsample']
}

grid_search =  GridSearchCV(

       estimator= full_pipeline,
       param_grid=param_grid,
       cv=5,
       scoring="recall",
       verbose=2
)


print ("Please wait ...")
grid_search.fit(x_train, y_train)


best_pipeline = grid_search.best_estimator_

print("/nThe best parametars :")
print (grid_search.best_params_)

y_pred_new = grid_search.predict(x_test)

print("\n=== New Classification Report ===")
print(classification_report(y_test , y_pred_new))

Please wait ...
Fitting 5 folds for each of 24 candidates, totalling 120 fits


c:\Program Files\Python39\lib\site-packages\sklearn\model_selection\_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Program Files\Python39\lib\site-packages\sklearn\model_selection\_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 380, in _score
    y_pred = method_caller(
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 90, in _cached_call
    result, _ = _get_response_values(
  File "c:\Program Files\Python39\lib\site-packages\sklearn\utils\_response.py", line 207, in _get_response_values
    ra

[CV] END classifier__class_weight=balanced, classifier__max_depth=10, classifier__n_estimators=100; total time=   6.2s


c:\Program Files\Python39\lib\site-packages\sklearn\model_selection\_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Program Files\Python39\lib\site-packages\sklearn\model_selection\_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 380, in _score
    y_pred = method_caller(
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 90, in _cached_call
    result, _ = _get_response_values(
  File "c:\Program Files\Python39\lib\site-packages\sklearn\utils\_response.py", line 207, in _get_response_values
    ra

[CV] END classifier__class_weight=balanced, classifier__max_depth=10, classifier__n_estimators=100; total time=   6.2s


...

[CV] END classifier__class_weight=balanced_subsample, classifier__max_depth=None, classifier__n_estimators=300; total time=  29.0s


c:\Program Files\Python39\lib\site-packages\sklearn\model_selection\_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Program Files\Python39\lib\site-packages\sklearn\model_selection\_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 380, in _score
    y_pred = method_caller(
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 90, in _cached_call
    result, _ = _get_response_values(
  File "c:\Program Files\Python39\lib\site-packages\sklearn\utils\_response.py", line 207, in _get_response_values
    ra

[CV] END classifier__class_weight=balanced_subsample, classifier__max_depth=None, classifier__n_estimators=300; total time=  29.2s


c:\Program Files\Python39\lib\site-packages\sklearn\model_selection\_validation.py:960: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Program Files\Python39\lib\site-packages\sklearn\model_selection\_validation.py", line 949, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 288, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 380, in _score
    y_pred = method_caller(
  File "c:\Program Files\Python39\lib\site-packages\sklearn\metrics\_scorer.py", line 90, in _cached_call
    result, _ = _get_response_values(
  File "c:\Program Files\Python39\lib\site-packages\sklearn\utils\_response.py", line 207, in _get_response_values
    ra

[CV] END classifier__class_weight=balanced_subsample, classifier__max_depth=None, classifier__n_estimators=300; total time=  28.8s
/nThe best parametars :
{'classifier__class_weight': 'balanced', 'classifier__max_depth': 10, 'classifier__n_estimators': 100}

=== New Classification Report ===
              precision    recall  f1-score   support

          No       0.91      0.70      0.79      1035
         Yes       0.49      0.81      0.61       374

    accuracy                           0.73      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.80      0.73      0.74      1409



In [23]:
import joblib


joblib.dump(best_pipeline, 'optimized_telecom_churn_model.pkl')

print("Done.")

Done.
